# 12 — Cone-Aware Auto-Seed for Tilted Detectors

The default Hough-circle BC seeder assumes Debye-Scherrer rings
are *circles* on the detector.  At non-zero tilt, the cone-detector
intersection is an **ellipse**, and the ellipse center is offset
from the true BC by

  Δ ≈ L_sd · sin(α) · tan²(2θ) / cos²(α)

— different rings have different offsets, so the per-ring center
sequence extrapolated to (2θ → 0) recovers the true BC.

This notebook is a synthetic POC of the cone-aware seed: forward-
project ring edge points through the v2 tilted-detector geometry
at known tilts, fit each ring as an ellipse, extrapolate the
centers, compare to the naive per-ring centroid (≈ what
Hough-circle voting gives in the small-arc limit).

The proof point: at 5° tilt the naive method biases by ~10 px (>
the LM basin width of 1.5 px); the cone-aware fit recovers BC to
~5 × 10⁻³ px.


In [1]:
import os, math
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import numpy as np
from typing import Tuple

def fit_ellipse(x: np.ndarray, y: np.ndarray) -> dict:
    """Fitzgibbon-Pilu-Fisher direct least-squares ellipse fit."""
    D = np.column_stack([x*x, x*y, y*y, x, y, np.ones_like(x)])
    S = D.T @ D
    C = np.zeros((6, 6))
    C[0, 2] = 2; C[2, 0] = 2; C[1, 1] = -1
    try:
        eig_vals, eig_vecs = np.linalg.eig(np.linalg.solve(S, C))
    except np.linalg.LinAlgError:
        return dict(ok=False)
    pos = np.argmax(eig_vals.real)
    a = eig_vecs[:, pos].real
    A_, B_, Cc_, D_, E_, F_ = a
    disc = B_*B_ - 4.0 * A_ * Cc_
    if abs(disc) < 1e-14:
        return dict(ok=False)
    h = (2.0 * Cc_ * D_ - B_ * E_) / disc
    k = (2.0 * A_ * E_ - B_ * D_) / disc
    return dict(ok=True, h=h, k=k)


def synth_ring(BC_y, BC_z, Lsd_um, ty_deg, tz_deg, two_theta_deg, p_x,
                n_points=360, jitter_px=0.5, rng=None):
    """Forward-project Debye-Scherrer cone of half-angle 2θ onto a tilted plane."""
    if rng is None: rng = np.random.default_rng(0)
    eta = np.linspace(0.0, 2.0 * np.pi, n_points, endpoint=False)
    tt = math.radians(two_theta_deg)
    cy_t, sy_t = math.cos(math.radians(ty_deg)), math.sin(math.radians(ty_deg))
    cz_t, sz_t = math.cos(math.radians(tz_deg)), math.sin(math.radians(tz_deg))
    ex_det = np.array([cz_t * cy_t, sz_t * cy_t, -sy_t])
    ey_det = np.array([-sz_t, cz_t, 0.0])
    ez_det = np.array([cz_t * sy_t, sz_t * sy_t, cy_t])
    p0 = np.array([Lsd_um, 0.0, 0.0])

    sx = math.cos(tt); sy = math.sin(tt) * np.sin(eta); sz = math.sin(tt) * np.cos(eta)
    cone = np.column_stack([np.full_like(sy, sx), sy, sz])
    denom = cone @ ex_det
    keep = np.abs(denom) > 1e-12
    t_num = float(p0 @ ex_det)
    t = np.where(keep, t_num / denom, 0.0)
    pts = t[:, None] * cone
    rel = pts - p0
    Y = (rel @ ey_det) / p_x + BC_y
    Z = (rel @ ez_det) / p_x + BC_z
    if jitter_px > 0:
        Y = Y + rng.normal(0, jitter_px, Y.shape)
        Z = Z + rng.normal(0, jitter_px, Z.shape)
    return Y[keep], Z[keep]


## Sweep over tilts and compare methods


In [2]:
BC_y_truth, BC_z_truth = 1024.0, 1024.0
Lsd_um = 900_000.0; p_x = 150.0
a_truth = 5.4116; lam_A = 0.197

hkl_sq = [3,4,8,11,12,16,19,20,24,27,32,35,36,40,43,44,48,51,52]
hkl_norm = np.sqrt(np.array(hkl_sq, dtype=np.float64))
keep = (lam_A * hkl_norm / (2.0 * a_truth)) < 0.95
hkl_norm = hkl_norm[keep]
tt_deg = 2.0 * np.degrees(np.arcsin(lam_A * hkl_norm / (2.0 * a_truth)))
print(f'rings: {len(tt_deg)} CeO2 rings (2θ {tt_deg[0]:.1f}–{tt_deg[-1]:.1f}°)')

rng = np.random.default_rng(0)
print(f'\n{"tilt (°)":>10s}  {"naive (px)":>12s}  {"cone-aware (px)":>16s}  {"improvement":>14s}')
for ty in (0.0, 1.0, 5.0, 10.0, 15.0):
    Y_list, Z_list = [], []
    for tt in tt_deg:
        Y, Z = synth_ring(BC_y_truth, BC_z_truth, Lsd_um, ty, 0.0,
                            float(tt), p_x, n_points=360, jitter_px=0.5, rng=rng)
        Y_list.append(Y); Z_list.append(Z)
    # Naive: per-ring centroid
    bc_y_naive = np.mean([Y.mean() for Y in Y_list])
    bc_z_naive = np.mean([Z.mean() for Z in Z_list])
    err_naive = math.hypot(bc_y_naive - BC_y_truth, bc_z_naive - BC_z_truth)
    # Cone-aware: ellipse fit + (tan²(2θ) → 0) extrapolation
    cy = []; cz = []
    for Y, Z in zip(Y_list, Z_list):
        ef = fit_ellipse(Y, Z)
        if ef['ok']:
            cy.append(ef['h']); cz.append(ef['k'])
        else:
            cy.append(np.nan); cz.append(np.nan)
    cy = np.array(cy); cz = np.array(cz)
    x_extrap = np.tan(np.radians(tt_deg)) ** 2
    good = np.isfinite(cy) & np.isfinite(cz)
    if good.sum() >= 2:
        py_y = np.polyfit(x_extrap[good], cy[good], 1)
        py_z = np.polyfit(x_extrap[good], cz[good], 1)
        bc_y_cone = py_y[1]; bc_z_cone = py_z[1]
        err_cone = math.hypot(bc_y_cone - BC_y_truth, bc_z_cone - BC_z_truth)
    else:
        err_cone = float('nan')
    impr = err_naive / err_cone if err_cone > 1e-6 else float('inf')
    print(f'  {ty:>10.1f}  {err_naive:>12.2f}  {err_cone:>16.3f}  {impr:>14.1f}×')


rings: 19 CeO2 rings (2θ 3.6–15.1°)

  tilt (°)    naive (px)   cone-aware (px)     improvement
         0.0          0.01             0.016             0.3×
         1.0          1.99             0.007           270.2×
         5.0         10.01             0.005          2178.5×
        10.0         20.43             0.033           618.1×
        15.0         31.71             0.122           259.7×


## What you should see

| Tilt | Naive err | Cone-aware err | Improvement |
|---|---|---|---|
| 0° | ~0.01 px | ~0.02 px | — |
| 1° | ~2 px | ~0.007 px | ~270× |
| 5° | ~10 px | ~0.005 px | ~2,000× |
| 10° | ~20 px | ~0.03 px | ~600× |
| 15° | ~32 px | ~0.12 px | ~260× |

The naive bias matches the closed-form
`L_sd · sin(α) · ⟨tan²(2θ)⟩ / cos²(α)` formula.  At zero tilt the
two methods agree (ellipse → circle).  At any tilt ≥ 1° the cone-
aware method recovers BC to sub-pixel level.

## What this is not (yet)

This is a **proof of concept**.  Wiring it into the production
[`seed.hough`](../midas_calibrate_v2/seed/hough.py) module
(replacing the circle-vote step with ellipse fit + extrapolation)
is mechanical: detect ring edge pixels with the existing
mask + ring-detection logic, group them by ring, fit each as an
ellipse, extrapolate.  The LM step is unchanged.

## See also

- [`dev/paper/runners/run_cone_aware_seed.py`](../dev/paper/runners/run_cone_aware_seed.py) — the runner this notebook is built from
- §"Cone-aware auto-seed for tilted detectors" in the paper
